# Flow Matching De-Haze Model

This notebook implements a de-haze model using flow matching and trains it on the SOTS dataset.

## 1. Imports & Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms as tfs
from torchvision.utils import save_image, make_grid
from tqdm import tqdm
import os
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr

# Configuration
data_dir = '../../../datasets/SOTS/outdoor'
output_dir = './outputs'
batch_size = 4
lr = 1e-4
num_epochs = 50
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Check if a GPU is available and set the device accordingly
device = "cpu"
# Check if a GPU is available and set the device accordingly
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator().type

print(f"Using {device} device")

## 2. Dataset Preparation

In [ ]:
class SOTSDataset(torch.utils.data.Dataset):
    def __init__(self, hazy_images, hazy_dir, clear_dir, transform=None):
        self.hazy_images = hazy_images
        self.hazy_dir = hazy_dir
        self.clear_dir = clear_dir
        self.transform = transform

    def __len__(self):
        return len(self.hazy_images)

    def __getitem__(self, idx):
        hazy_path = os.path.join(self.hazy_dir, self.hazy_images[idx])
        clear_path = os.path.join(self.clear_dir, self.hazy_images[idx].split('_')[0] + '.png')
        hazy = Image.open(hazy_path).convert('RGB')
        clear = Image.open(clear_path).convert('RGB')
        if self.transform:
            hazy = self.transform(hazy)
            clear = self.transform(clear)
        # Ensure the dataset returns exactly two values
        return hazy, clear

# Transforms
transform = tfs.Compose([
    tfs.Resize((256, 256)),
    tfs.ToTensor(),
    tfs.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Dataset split
hazy_dir = os.path.join(data_dir, 'hazy')
clear_dir = os.path.join(data_dir, 'clear')
hazy_images = os.listdir(hazy_dir)
train_images, test_images = train_test_split(hazy_images, test_size=0.2, random_state=42)

# Dataloaders
train_dataset = SOTSDataset(train_images, hazy_dir, clear_dir, transform=transform)
test_dataset = SOTSDataset(test_images, hazy_dir, clear_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
# Display some sample hazy and clear images side-by-side
# Denormalization function
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

# Update show_samples function
def show_samples(dataset, num_samples=4):
    mean = [0.5, 0.5, 0.5]
    std = [0.5, 0.5, 0.5]
    fig, axes = plt.subplots(num_samples, 2, figsize=(8, 4 * num_samples))
    for i in range(num_samples):
        hazy, clear = dataset[i]
        hazy = denormalize(hazy.clone(), mean, std).permute(1, 2, 0).numpy()
        clear = denormalize(clear.clone(), mean, std).permute(1, 2, 0).numpy()
        hazy = np.clip(hazy, 0, 1)  # Ensure values are in [0, 1] range
        clear = np.clip(clear, 0, 1)  # Ensure values are in [0, 1] range
        axes[i, 0].imshow(hazy)
        axes[i, 0].set_title('Hazy Image')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(clear)
        axes[i, 1].set_title('Clear Image')
        axes[i, 1].axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_dataset)  # Show samples from training set

## 3. Model Definition

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super(UNet, self).__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1),
                nn.ReLU(inplace=True)
            )

        def up_block(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2),
                nn.ReLU(inplace=True)
            )

        self.enc1 = conv_block(in_channels, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)
        self.enc4 = conv_block(256, 512)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = conv_block(512, 1024)

        self.up4 = up_block(1024, 512)
        self.dec4 = conv_block(1024, 512)
        self.up3 = up_block(512, 256)
        self.dec3 = conv_block(512, 256)
        self.up2 = up_block(256, 128)
        self.dec2 = conv_block(256, 128)
        self.up1 = up_block(128, 64)
        self.dec1 = conv_block(128, 64)

        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))

        bottleneck = self.bottleneck(self.pool(enc4))

        dec4 = self.up4(bottleneck)
        dec4 = self.dec4(torch.cat([dec4, enc4], dim=1))
        dec3 = self.up3(dec4)
        dec3 = self.dec3(torch.cat([dec3, enc3], dim=1))
        dec2 = self.up2(dec3)
        dec2 = self.dec2(torch.cat([dec2, enc2], dim=1))
        dec1 = self.up1(dec2)
        dec1 = self.dec1(torch.cat([dec1, enc1], dim=1))

        return torch.tanh(self.final(dec1))

model = UNet().to(device)

## 4. Training and Evaluation Loop

In [ ]:
# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for hazy, clear in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        hazy, clear = hazy.to(device), clear.to(device)

        # Forward pass
        outputs = model(hazy)
        loss = criterion(outputs, clear)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss:.4f}")

    # Save model checkpoint
    torch.save(model.state_dict(), os.path.join(output_dir, f"unet_epoch_{(epoch % 5) + 1}.pth"))

    # Evaluation loop
    model.eval()
    psnr_values = []
    with torch.no_grad():
        for idx, (hazy, clear) in enumerate(tqdm(test_loader, desc="Evaluating")):
            hazy, clear = hazy.to(device), clear.to(device)

            # Forward pass
            outputs = model(hazy)

            outputs_np = outputs.squeeze(0).permute(1, 2, 0).cpu().numpy()
            clear_np = clear.squeeze(0).permute(1, 2, 0).cpu().numpy()
            hazy_np = hazy.squeeze(0).permute(1, 2, 0).cpu().numpy()

            # Save RGB PNG files (convert to 8-bit)
            if idx < 10:  # Save only the first 10 samples
                comparison = np.concatenate([hazy_np, outputs_np, clear_np], axis=1)
                comparison = np.clip(comparison * 255, 0, 255).astype(np.uint8)  # Scale to [0, 255]
                Image.fromarray(comparison).save(
                    os.path.join(output_dir, f"comparison_{idx+1}.png"),
                    format="PNG"
                )

            # Calculate PSNR
            psnr_values.append(compare_psnr(clear_np, outputs_np, data_range=1))

    print(f"Average PSNR: {np.mean(psnr_values):.4f}")